<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/5_transformer_model/5_3_baseline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5_3_baseline_model

## Introducción y Resumen

El objetivo de esta notebook es entrenar, ajustar y evaluar de forma rigurosa un modelo Transformer para la predicción de retornos financieros intradía, utilizando ventanas temporales de datos minuto a minuto (OHLCV + factores alpha), incorporando validación cruzada y tuneo de hiperparámetros, con el fin de optimizar su capacidad predictiva y comparar su desempeño bajo distintas configuraciones del modelo.

En términos operativos, la notebook tiene como propósito:

- Entrenar el modelo sobre datos intradía estructurados en ventanas temporales.
- Ajustar hiperparámetros clave (arquitectura, learning rate, regularización, batch size, etc.) mediante procedimientos sistemáticos de tuning.
- Evaluar el desempeño con métricas de error y direccionalidad, asegurando consistencia entre folds.
- Seleccionar configuraciones óptimas del Transformer para su posterior análisis y comparación con otros modelos.





## 0. Configuración del Entorno


### 0.1. Instalación de librerías


### 0.2. Importación de librerías


In [1]:
# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning y utilidades
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
import sklearn, scipy, numpy #, optuna

import joblib

# ==============================
# Configuración general
# ==============================
warnings.filterwarnings("ignore")

import time

import sys, platform, lightgbm as lgb
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [2]:
print("python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
#print("optuna:", optuna.__version__)
#print("xgboost:", xgb.__version__)
#print("lightgbm:", lgb.__version__)

# CatBoost usa la clase para exponer versión
#print("catboost:", catboost.__version__)

python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
numpy: 2.0.2
scipy: 1.16.3
sklearn: 1.6.1


### 0.3. Acceso a Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.4. Comprobación de uso de RAM

In [4]:
import psutil

def ram_usage():
    ram = psutil.virtual_memory()
    used = ram.used / (1024**3)
    free = ram.available / (1024**3)
    total = ram.total / (1024**3)

    print(f"RAM total:      {total:.2f} GB")
    print(f"RAM usada:      {used:.2f} GB")
    print(f"RAM disponible: {free:.2f} GB")

## **1. Carga de datos**

### **1.1. De datasets**

#### 1.1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [5]:
def load_data(data: str):

    data_path = f'{drive_path}/5_transformer_model/5_0_k_folds/fold_{fold}/{data}_{fold}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [6]:
#mnq_train = {}
#mnq_valid = {}
#mnq_test  = {}

#for k in k_folds:
#    print(f'Cargando datos de Fold {k}..')
#    mnq_train[k] = load_data(str(k), 'train')
#    mnq_valid[k] = load_data(str(k), 'valid')
#    mnq_test[k]  = load_data(str(k), 'test')

#### 1.1.2. Información de datasets


In [7]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [8]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

#### 1.1.3. Carga de listado de features por ventana de tiempo

In [9]:
import json
# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_90 = features_dict["features_to_90"]

In [10]:
print(f'Listado de features para 90min: {features_to_90}')

Listado de features para 90min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


## 2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`

In [11]:
#Lista de K folds
k_folds = [1, 2, 3, 4, 5]

### 2.0. Funciones

#### 2.0.1. Función para cargar ventanas

In [12]:
def load_windows_and_scaler(k: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_train_sc_{k}.npz'
    path_valid  = f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_valid_sc_{k}.npz'
    path_test   = f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_test_sc_{k}.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/5_transformer_model/5_2_k_scaler/global_scaler.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    print(f'\tX_train_sc_{k} e y_train_{k} extraídos correctamente')
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    print(f'\tX_valid_sc_{k} e y_valid_{k} extraídos correctamente')
    X_test,  y_test  = data_test["X"],  data_test["y"]
    print(f'\tX_test_sc_{k} e y_test_{k} extraídos correctamente')

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### 2.0.2. Función para revisar información de ventanas

In [13]:
def xy_info(k, X_train, y_train, X_valid, y_valid, X_test, y_test, silent=False):
    import numpy as np
    import psutil

    if not silent:
        print(f"Información de {k}:")
        print("----------------------------------------")

    # Memoria total
    total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)

    def print_set_info(nombre, X, y):
        if silent:
            return  # No imprimir nada

        n_samples = X.shape[0]
        size_X_gb = X.nbytes / (1024 ** 3)
        size_y_gb = y.nbytes / (1024 ** 3)
        total_gb = size_X_gb + size_y_gb
        perc_ram = (total_gb / total_ram_gb) * 100
        y_flat = np.ravel(y)

        print(f"Set de {nombre}:")
        print(f"\t{n_samples} ventanas")
        print(f"\tTamaño X: {size_X_gb:.3f} GB")
        print(f"\tTamaño y: {size_y_gb:.6f} GB")
        print(f"\tTOTAL: {total_gb:.3f} GB → {perc_ram:.1f}% RAM\n")

    # Mostrar info solo si silent=False
    print_set_info("entrenamiento", X_train, y_train)
    print_set_info("validación",    X_valid, y_valid)
    print_set_info("testeo",        X_test,  y_test)

    # Pesos = cantidad de ventanas
    w_train = X_train.shape[0]
    w_valid = X_valid.shape[0]
    w_test  = X_test.shape[0]

    return w_train, w_valid, w_test


### 2.1 Carga de ventanas

In [14]:
#Para verificar el formato de lo guardado.
#for k in k_folds:
#    print(f'Fold {k}:')
#    print('\tTrain:\t', np.load(f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_train_sc_{k}.npz').files)
#    print('\tValid:\t', np.load(f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_valid_sc_{k}.npz').files)
#    print('\tTest:\t',np.load(f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_test_sc_{k}.npz').files)

In [15]:
ram_usage()

RAM total:      52.96 GB
RAM usada:      1.53 GB
RAM disponible: 50.79 GB


In [16]:
# Diccionarios para almacenar datos escalados por fold
X_train_sc = {}
y_train_sc = {}
X_valid_sc = {}
y_valid_sc = {}
X_test_sc  = {}
y_test_sc  = {}
scalers    = {}

for k in k_folds:
    print(f'Fold {k}:')

    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler = load_windows_and_scaler(k)

    # Guardar todo en diccionarios
    X_train_sc[k] = X_train
    y_train_sc[k] = y_train

    X_valid_sc[k] = X_valid
    y_valid_sc[k] = y_valid

    X_test_sc[k]  = X_test
    y_test_sc[k]  = y_test

    scalers[k] = scaler

    print(f"  - Datos escalados cargados y almacenados en diccionarios.")
    print("-" * 40)

Fold 1:
	X_train_sc_1 e y_train_1 extraídos correctamente
	X_valid_sc_1 e y_valid_1 extraídos correctamente
	X_test_sc_1 e y_test_1 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
----------------------------------------
Fold 2:
	X_train_sc_2 e y_train_2 extraídos correctamente
	X_valid_sc_2 e y_valid_2 extraídos correctamente
	X_test_sc_2 e y_test_2 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
----------------------------------------
Fold 3:
	X_train_sc_3 e y_train_3 extraídos correctamente
	X_valid_sc_3 e y_valid_3 extraídos correctamente
	X_test_sc_3 e y_test_3 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
----------------------------------------
Fold 4:
	X_train_sc_4 e y_train_4 extraídos correctamente
	X_valid_sc_4 e y_valid_4 extraídos correctamente
	X_test_sc_4 e y_test_4 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
-------------

In [17]:
#Como accedeR:
#Xtr = X_train_sc[3]   # X_train del fold 3
#ytr = y_train_sc[3]

In [18]:
ram_usage()

RAM total:      52.96 GB
RAM usada:      6.35 GB
RAM disponible: 45.97 GB


In [19]:
pesos_folds = {}

for k in k_folds:
    w_train, w_valid, w_test = xy_info(
        k,
        X_train_sc[k],
        y_train_sc[k],
        X_valid_sc[k],
        y_valid_sc[k],
        X_test_sc[k],
        y_test_sc[k],
        silent=True   # evita imprimir
    )

    pesos_folds[k] = {
        "w_train": w_train,
        "w_valid": w_valid,
        "w_test":  w_test,
    }


In [20]:
pesos_folds

{1: {'w_train': 124279, 'w_valid': 24898, 'w_test': 27852},
 2: {'w_train': 149177, 'w_valid': 24898, 'w_test': 27852},
 3: {'w_train': 174075, 'w_valid': 24898, 'w_test': 27852},
 4: {'w_train': 198973, 'w_valid': 24898, 'w_test': 27852},
 5: {'w_train': 223871, 'w_valid': 24898, 'w_test': 27852}}

In [21]:
ram_usage()

RAM total:      52.96 GB
RAM usada:      6.35 GB
RAM disponible: 45.97 GB


## 3. Dataset de Métricas

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

### 3.1. Función para cargar métricas o generar dataset

In [22]:
drive_path

'/content/drive/MyDrive/neural_profit'

In [23]:
def load_metrics(subcarpeta: str, data: str):
    data_path = f'{drive_path}/5_transformer_model/{subcarpeta}/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [24]:
def metrics_verify(subcarpeta: str, data: str) -> bool:
    data_path = f'{drive_path}/5_transformer_model/{subcarpeta}/{data}.parquet'
    return os.path.exists(data_path)


In [25]:
def load_or_create_metrics (subcarpeta: str, data:str):
  if metrics_verify(subcarpeta, data):
      print(f"Las métricas existen y son almacenadas en {data[2:len(data)]}")
      model_metrics = load_metrics(subcarpeta, data)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[2:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [26]:
baseline_folds_metrics, flag_baseline_folds_metrics = load_or_create_metrics("5_3_baseline_model", "0_baseline_folds_metrics")
baseline_metrics, flag_baseline_metrics = load_or_create_metrics("5_3_baseline_model", "1_baseline_metrics")

Las métricas no existen. Se crea el dataset baseline_folds_metrics para almacenar las métricas
Las métricas no existen. Se crea el dataset baseline_metrics para almacenar las métricas


In [27]:
baseline_folds_metrics

,RMSE,MAE,R2,SMAPE,DirAcc


### 3.2. Función para guardar métricas

In [28]:
def save_metrics (metrics,  subcarpeta: str, metrics_name: str):
  #metrics_path = f"{drive_path}/5_transformer_model/5_3_model_transformer/{subcarpeta}/{metrics_name}.parquet"
  metrics_path = f"{drive_path}/5_transformer_model/{subcarpeta}/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

### 3.3. Función para calcular las métricas

In [29]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [30]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

# Preparación para entrenamiento base de Transformers

## **4. Re-formateo más Encoder mínimo**

### **4.1. Helper: de 2D (aplanado) a 3D (B, T, F)**

Este bloque define una función auxiliar utilizada para **re-formatear las ventanas de datos** desde una representación 2D a la representación 3D requerida por el modelo.

- **Entrada**:
  - `X_flat`: matriz 2D con forma *(N, window_size × n_features)*.
  - `window_size`: longitud temporal de la ventana.
  - `n_features`: cantidad de features por paso temporal.

- **Funcionamiento**:
  - Verifica que la entrada sea efectivamente una matriz 2D.
  - Valida la consistencia dimensional comprobando que  
    `window_size × n_features == X_flat.shape[1]`.
  - Reconvierte los datos al formato *(N, window_size, n_features)* mediante `reshape`.

- **Salida**:
  - Un arreglo 3D listo para ser utilizado como entrada del modelo.

Este helper se utiliza para **cada conjunto de datos (train, valid y test)**.  
En este proyecto se emplea `window_size = 90` y `n_features_90 = 12`, asegurando que cada ventana temporal esté correctamente estructurada antes del entrenamiento.

In [31]:
import numpy as np
import torch
import torch.nn as nn
import math

def reshape_windows(X_flat: np.ndarray, window_size: int, n_features: int) -> np.ndarray:
    """
    Convierte X de (N, window_size * n_features) a (N, window_size, n_features).
    Valida la consistencia del producto.
    """
    assert X_flat.ndim == 2, "Se esperaba X_flat con 2D (N, T*F)."
    N, TF = X_flat.shape
    assert window_size * n_features == TF, (
        f"Inconsistencia: {window_size} * {n_features} != {TF}"
    )
    return X_flat.reshape(N, window_size, n_features)

En este bloque se definen los parámetros estructurales de entrada del modelo:

- `window_size = 90`  
  Establece la longitud de la ventana temporal, es decir, la cantidad de minutos consecutivos utilizados como entrada para cada muestra.

- `features_base`  
  Contiene las variables OHLCV básicas del mercado: *open, high, close, low y volume*.

- `features_90`  
  Se construye combinando las variables base con los factores adicionales definidos en `features_to_90`, conformando el conjunto completo de features utilizadas por el modelo.

- `n_features_90`  
  Representa la cantidad total de features por paso temporal y se obtiene como la longitud de `features_90`.

Este bloque permite **verificar explícitamente** el conjunto de features y su cardinalidad, asegurando coherencia dimensional con la configuración del modelo y las funciones de re-formateo de ventanas.

In [32]:
window_size = 90
features_base = ['open','high','close','low','volume']
features_90 = features_base + features_to_90
n_features_90 = len (features_90)
print(f'features_90:\t\t {features_90}')
print(f'n_features_90:\t {n_features_90}')

features_90:		 ['open', 'high', 'close', 'low', 'volume', 'ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']
n_features_90:	 12


**Re-formateo de ventanas por fold y liberación de memoria**

Este bloque se encarga de **transformar las ventanas de entrada desde formato 2D a formato 3D**, de acuerdo con la configuración definida previamente (`window_size = 90` y `n_features_90 = 12`), y de **optimizar el uso de memoria** durante el procesamiento por fold.

- Para cada *fold* en `k_folds`:
  - Las matrices escaladas `X_train_sc[k]`, `X_valid_sc[k]` y `X_test_sc[k]`, originalmente en formato  
    *(N, window_size × n_features_90)*, se convierten al formato requerido por el modelo:  
    *(N, window_size, n_features_90)* mediante la función `reshape_windows`.
  - Los datos re-formateados se almacenan en los diccionarios `Xtr`, `Xva` y `Xte`, indexados por fold.

- Una vez completado el re-formateo de cada fold:
  - Se eliminan explícitamente las matrices 2D originales para evitar duplicación innecesaria de datos en memoria.
  - Se invoca el recolector de basura (`gc.collect()`) para liberar RAM de forma inmediata.

El objetivo principal es asegurar que cada conjunto de datos esté correctamente estructurado para el entrenamiento del modelo Transformer, manteniendo un consumo de memoria controlado durante el procesamiento de múltiples folds.

In [33]:
import gc

Xtr = {}
Xva = {}
Xte = {}

for k in k_folds:
    Xtr[k] = reshape_windows(X_train_sc[k], window_size, n_features_90)
    Xva[k] = reshape_windows(X_valid_sc[k], window_size, n_features_90)
    Xte[k] = reshape_windows(X_test_sc[k],  window_size, n_features_90)

    # Liberar las matrices 2D de este fold
    del X_train_sc[k], X_valid_sc[k], X_test_sc[k]
    gc.collect()

    print(f'Fold {k} re-shape completo y 2D liberado')

Fold 1 re-shape completo y 2D liberado
Fold 2 re-shape completo y 2D liberado
Fold 3 re-shape completo y 2D liberado
Fold 4 re-shape completo y 2D liberado
Fold 5 re-shape completo y 2D liberado


**Verificación de dimensiones de entrada por fold**

Este bloque define una función auxiliar destinada a **verificar las dimensiones de los datos de entrada** del modelo para cada fold.

- La función itera sobre los *folds* definidos en `k_folds`.
- Para cada fold:
  - Muestra de forma ordenada los *shapes* de los conjuntos **train**, **valid** y **test**.
  - Verifica que cada conjunto se encuentre en formato 3D, consistente con la estructura  
    *(n_samples, window_size, n_features)* requerida por el modelo.

- La salida se presenta en forma tabular, facilitando la inspección visual y la detección temprana de inconsistencias dimensionales entre folds o conjuntos de datos.

El objetivo principal es confirmar que el re-formateo de las ventanas se haya realizado correctamente antes de proceder al entrenamiento y evaluación del modelo.


In [34]:
def mostrar_shapes_folds_simple(k_folds, Xtr, Xva, Xte):
    for k in k_folds:
        print(f"\nShapes del Fold {k}")
        print(f"{'Set':<10}{'Shape (3D)':<25}")
        print("-" * 40)

        filas = [
            ("Train", Xtr[k].shape),
            ("Valid", Xva[k].shape),
            ("Test",  Xte[k].shape),
        ]

        for nombre, shape_3d in filas:
            print(f"{nombre:<10}{str(shape_3d):<25}")

In [35]:
mostrar_shapes_folds_simple(k_folds, Xtr, Xva, Xte)


Shapes del Fold 1
Set       Shape (3D)               
----------------------------------------
Train     (124279, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 2
Set       Shape (3D)               
----------------------------------------
Train     (149177, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 3
Set       Shape (3D)               
----------------------------------------
Train     (174075, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 4
Set       Shape (3D)               
----------------------------------------
Train     (198973, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 5
Set       Shape (3D)               
----------------------------------------
Train     (223871, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852

### **4.2. Encoder: backbone + posición + TransformerEncoder**

Este bloque define el **encoder temporal del modelo**, que actúa como *backbone* y es responsable de transformar las ventanas de entrada en **representaciones latentes ricas por paso temporal**, capturando dependencias temporales y relaciones entre features.


#### **4.2.1. Codificación posicional sinusoidal**


La clase `SinusoidalPositionalEncoding` implementa una **codificación posicional determinística**, basada en funciones seno y coseno, tal como fue introducida en el Transformer original.

- Genera una matriz de posiciones de tamaño `(max_len, d_model)` donde:
  - Las posiciones pares usan funciones seno.
  - Las posiciones impares usan funciones coseno.
- Esta codificación:
  - No tiene parámetros entrenables.
  - Permite al modelo incorporar información sobre el **orden temporal** dentro de la ventana.
- Se registra como *buffer* (`register_buffer`), por lo que:
  - Se mueve automáticamente a CPU/GPU.
  - No se actualiza durante el entrenamiento.

En el `forward`, la codificación posicional se **suma** a los embeddings de entrada:
- Entrada: `(B, T, D)`
- Salida: `(B, T, D)`

In [36]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D)
        T = x.size(1)
        return x + self.pe[:, :T]

#### **4.2.2. Encoder de series temporales (`TimeSeriesEncoder`)**

La clase `TimeSeriesEncoder` implementa el **encoder completo**, compuesto por tres bloques principales:

1. **Proyección de entrada (backbone lineal)**  
   - Convierte las features originales de cada paso temporal desde:
     ```
     (B, T, F) → (B, T, d_model)
     ```
   - Permite trabajar en un espacio latente de mayor capacidad (`d_model`).

2. **Codificación posicional**  
   - Añade información explícita de posición temporal a cada embedding.
   - Es fundamental para que el Transformer distinga el orden dentro de la ventana.

3. **Transformer Encoder**  
   - Conformado por `num_layers` capas apiladas de `TransformerEncoderLayer`.
   - Cada capa incluye:
     - Multi-Head Self-Attention (`nhead`)
     - Feedforward interno (`dim_feedforward`)
     - Dropout y normalización (*pre-norm*, `norm_first=True`)
   - Opera con `batch_first=True`, manteniendo el formato `(B, T, D)`.

Finalmente, se aplica un `Dropout` adicional como regularización.

---

**Inicialización**

- La capa de proyección (`input_proj`) se inicializa con **Xavier Uniform**, asegurando estabilidad al inicio del entrenamiento.
- El sesgo se inicializa en cero.

---

**Salida del encoder**

El `forward` del encoder devuelve: `(B, T, d_model)`


Es decir, un **embedding contextualizado por cada paso temporal**, que posteriormente será utilizado por el bloque de *pooling* y la cabeza de regresión.


In [37]:
class TimeSeriesEncoder(nn.Module):
    """
    Proyección a d_model + PositionalEncoding + TransformerEncoder (sin cabeza).
    Devuelve embeddings por paso temporal: (B, T, d_model)
    """
    def __init__(
        self,
        input_dim: int,
        d_model: int = 128,
        #Hiperparámetros del modelo
        nhead: int = 8,
        num_layers: int = 2,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
        activation: str = "gelu",
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = SinusoidalPositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=activation,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)

        # init
        nn.init.xavier_uniform_(self.input_proj.weight)
        nn.init.zeros_(self.input_proj.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)
        z = self.input_proj(x)    # (B, T, D)
        z = self.pos_encoder(z)   # (B, T, D)
        z = self.encoder(z)       # (B, T, D)
        z = self.dropout(z)       # (B, T, D)
        return z

El objetivo principal es extraer representaciones temporales profundas y contextualizadas de cada ventana intradía, que sirvan como base para la predicción de retornos futuros.

#### **4.2.3. Inicialización de encoders por fold y asignación de dispositivo**


Este bloque se encarga de **instanciar el encoder del modelo para cada fold**, asegurando independencia entre entrenamientos y una correcta gestión del dispositivo de cómputo.

- Se detecta automáticamente el dispositivo disponible:
  - `cuda` si hay GPU disponible.
  - `cpu` en caso contrario.

- Se crea un diccionario `encoders` donde:
  - Cada *fold* posee su **propia instancia** de `TimeSeriesEncoder`.
  - Esto evita compartir pesos entre folds y garantiza aislamiento experimental durante la validación cruzada.

- Para cada fold:
  - Se inicializa el encoder con:
    - `input_dim = n_features_90`, correspondiente al número de features por paso temporal (12).
    - Hiperparámetros arquitectónicos fijados en esta etapa (`d_model`, `nhead`, `num_layers`), los cuales serán candidatos a ajuste en etapas posteriores de *hyperparameter tuning*.
  - El modelo se traslada explícitamente al dispositivo definido (`.to(device)`).

El objetivo principal es disponer de un encoder independiente y correctamente configurado para cada fold, permitiendo entrenamientos reproducibles y comparables dentro del esquema de validación cruzada.


In [38]:
baseline_params = {
    #De arquitectura
    "dropout": 0.1,
    "d_model": 128,
    "n_layers": 2,
    "n_heads": 8,
    "ff_mult": 2,
    "activation": "gelu",
    "pooling": "mean",
    "head_dropout": 0.1,

    #De entrenamiento
    "max_epochs": 50,
    "patience": 8,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,
  }


In [39]:
baseline_params

{'dropout': 0.1,
 'd_model': 128,
 'n_layers': 2,
 'n_heads': 8,
 'ff_mult': 2,
 'activation': 'gelu',
 'pooling': 'mean',
 'head_dropout': 0.1,
 'max_epochs': 50,
 'patience': 8,
 'lr': 0.0003,
 'weight_decay': 0.0001,
 'grad_clip': 1.0}

In [40]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Diccionario de encoders por fold
encoders = {}

for k in k_folds:
    encoders[k] = TimeSeriesEncoder(
        input_dim=n_features_90,
        # HIPERPARÁMETROS TUNEADOS Y DIJOS DE heavy_best_params
        d_model = baseline_params['d_model'],
        nhead = baseline_params['n_heads'],
        num_layers = baseline_params['n_layers'],
        dim_feedforward = baseline_params['d_model']*baseline_params['ff_mult'],
        dropout = baseline_params['dropout'],          # default
        activation = baseline_params['activation'],    # default
    ).to(device)
    print(f"Baseline encoder creado para fold {k}")

Baseline encoder creado para fold 1
Baseline encoder creado para fold 2
Baseline encoder creado para fold 3
Baseline encoder creado para fold 4
Baseline encoder creado para fold 5


#### **4.4.4. Verificación del funcionamiento del encoder por fold**

Este bloque define y ejecuta una función de **validación operativa del encoder**, cuyo objetivo es comprobar que el modelo procesa correctamente los datos de entrada y produce salidas con las dimensiones esperadas.

La función `verificar_encoder` recibe como entrada un *fold* específico, los datos re-formateados (`Xtr`, `Xva`, `Xte`) y el diccionario de encoders, y realiza los siguientes pasos:

1. **Selección del dispositivo**  
   Detecta automáticamente si se utilizará GPU (`cuda`) o CPU, garantizando coherencia con el encoder previamente instanciado.

2. **Carga de ventanas 3D**  
   Extrae los conjuntos *train*, *valid* y *test* correspondientes al fold, ya estructurados en formato: `(n_samples, window_size, n_features)`

3. **Creación de mini-batches de inspección**  
    - Selecciona las primeras 64 muestras de cada conjunto.
    - Las convierte a tensores `torch.float32` y las traslada al dispositivo.
    - No se busca entrenar, solo verificar el flujo de datos.

4. **Selección del encoder del fold**  
  Recupera la instancia de `TimeSeriesEncoder` asociada al fold, garantizando consistencia entre datos y modelo.

5. **Ejecución del encoder en modo inferencia**  
    - Ejecuta el encoder dentro de un bloque `torch.no_grad()`.
    - Evita el cálculo de gradientes y reduce el consumo de memoria.
    - Imprime la forma de la salida para cada conjunto.

La salida esperada del encoder es: `(B, T, d_model)`

In [41]:
def verificar_encoder(fold, Xtr, Xva, Xte, encoders):
    print(f"\n=== Fold {fold} ===")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # ---------- 1) Cargar ventanas 3D ----------
    Xtr_k = Xtr[fold]
    Xva_k = Xva[fold]
    Xte_k = Xte[fold]

    # ---------- 2) Mini-batches para inspección ----------
    xb_tr = torch.tensor(Xtr_k[:64], dtype=torch.float32).to(device)
    xb_va = torch.tensor(Xva_k[:64], dtype=torch.float32).to(device)
    xb_te = torch.tensor(Xte_k[:64], dtype=torch.float32).to(device)

    # ---------- 3) Encoder del fold ----------
    encoder = encoders[fold]

    # ---------- 4) Pares para inspección ----------
    pairs = [
        (xb_tr, encoder, f"Fold{fold}-train"),
        (xb_va, encoder, f"Fold{fold}-valid"),
        (xb_te, encoder, f"Fold{fold}-test"),
    ]

    # ---------- 5) Ejecutar encoder ----------
    for xb, enc, tag in pairs:
        with torch.no_grad():
            z = enc(xb)
        print(tag, "→", z.shape)



El siguiente bucle aplica esta verificación a todos los folds, confirmando que:

- El encoder acepta correctamente las entradas 3D.
- No existen inconsistencias dimensionales entre folds.
- El backbone del modelo está correctamente configurado antes del entrenamiento.

El objetivo principal es validar de forma temprana la compatibilidad entre los datos y el encoder, evitando errores silenciosos o fallos costosos durante la etapa de entrenamiento y tuneo de hiperparámetros.

In [42]:
for k in k_folds:
  verificar_encoder(k, Xtr, Xva, Xte, encoders)


=== Fold 1 ===
Fold1-train → torch.Size([64, 90, 128])
Fold1-valid → torch.Size([64, 90, 128])
Fold1-test → torch.Size([64, 90, 128])

=== Fold 2 ===
Fold2-train → torch.Size([64, 90, 128])
Fold2-valid → torch.Size([64, 90, 128])
Fold2-test → torch.Size([64, 90, 128])

=== Fold 3 ===
Fold3-train → torch.Size([64, 90, 128])
Fold3-valid → torch.Size([64, 90, 128])
Fold3-test → torch.Size([64, 90, 128])

=== Fold 4 ===
Fold4-train → torch.Size([64, 90, 128])
Fold4-valid → torch.Size([64, 90, 128])
Fold4-test → torch.Size([64, 90, 128])

=== Fold 5 ===
Fold5-train → torch.Size([64, 90, 128])
Fold5-valid → torch.Size([64, 90, 128])
Fold5-test → torch.Size([64, 90, 128])


Los resultados confirman que el **encoder funciona correctamente en todos los folds**.

Para los conjuntos de *train*, *valid* y *test*, el modelo recibe ventanas de longitud **90** y produce embeddings de dimensión **128** por paso temporal, manteniendo de forma consistente la estructura esperada `(batch, window_size, d_model)`.

Esto valida que:
- El re-formateo de las ventanas es correcto.
- La proyección de features y la codificación posicional están correctamente configuradas.
- El *backbone* del Transformer está listo para el entrenamiento y el posterior tuneo de hiperparámetros.


## **5. Pooling (Sin cambiar enconder)**

En esta etapa se introduce el **mecanismo de pooling temporal**, cuyo objetivo es **colapsar la dimensión temporal** de la salida del encoder sin alterar su arquitectura ni sus pesos.


- El encoder produce una salida de forma `(B, T, d_model)` donde cada paso temporal contiene un embedding contextualizado.

- El pooling se aplica **posteriormente al encoder**, transformando la salida a: `(B, d_model)` obteniendo una representación fija por ventana.

Este diseño permite:
- Mantener el encoder como *backbone* reutilizable.
- Evaluar distintos esquemas de agregación temporal sin reentrenar ni redefinir el encoder.
- Comparar el impacto del pooling en el desempeño predictivo del modelo.

- Los métodos de pooling típicos (simples y que no requieren modificar el encoder) incluyen:
  - **Mean pooling**: promedio sobre la dimensión temporal.
  - **Last pooling**: selección del último paso temporal.
  - **Attention pooling**: combinación ponderada aprendida de los pasos temporales.

El objetivo principal es extraer una representación global de cada ventana temporal a partir de los embeddings del encoder, preparándola para su uso en la cabeza de regresión (regresion head) del modelo.

#### **5.1. Pooling Temporal**

Este bloque define un módulo de **pooling temporal**, encargado de **agregar la dimensión temporal** de la salida del encoder sin modificar su estructura ni sus parámetros.

- **Entrada**:
  - Tensor `z` con forma `(B, T, D)`, correspondiente a los embeddings temporales generados por el encoder.

- **Modos soportados**:
  - **`mean`**: realiza el promedio sobre la dimensión temporal, produciendo una representación global que resume toda la ventana.
  - **`last`**: selecciona el embedding del último paso temporal, representando el estado final de la secuencia.

- **Salida**:
  - Tensor con forma `(B, D)`, adecuado para ser utilizado por la cabeza de regresión del modelo.

Este enfoque permite **comparar distintas estrategias de agregación temporal** manteniendo fijo el encoder, facilitando el análisis del impacto del pooling en el desempeño predictivo.

In [43]:
import torch
import torch.nn as nn

class TemporalPooling(nn.Module):
    def __init__(self, mode: str = "mean"):
        super().__init__()
        assert mode in ("mean", "last"), "Soportado: 'mean' o 'last'"
        self.mode = mode

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # z: (B, T, D)
        if self.mode == "mean":
            return z.mean(dim=1)      # (B, D)
        else:  # "last"
            return z[:, -1, :]        # (B, D)

####**5.2. Verificación del pooling temporal por fold**

Este bloque se utiliza para **validar el funcionamiento de los mecanismos de pooling temporal**, aplicados sobre la salida del encoder, sin modificar su arquitectura.

- Se instancian dos estrategias de pooling:
  - `pool_mean`: promedio sobre la dimensión temporal.
  - `pool_last`: selección del último paso temporal.

- Para cada *fold*:
  1. Se cargan las ventanas 3D de *train*, *valid* y *test* desde los diccionarios correspondientes.
  2. Se construyen mini-batches de 64 muestras y se trasladan al dispositivo de cómputo disponible.
  3. Se recupera el encoder asociado al fold.
  4. Cada mini-batch se procesa en modo inferencia (`torch.no_grad()`):
     - Primero por el encoder, obteniendo embeddings con forma `(B, T, d_model)`.
     - Luego por ambos métodos de pooling, colapsando la dimensión temporal a `(B, d_model)`.

- Finalmente, se imprimen las dimensiones resultantes para cada conjunto y método de pooling, permitiendo verificar:
  - La coherencia dimensional de las salidas.
  - La correcta integración del pooling con el encoder.

El objetivo principal es confirmar que las estrategias de pooling temporal generan representaciones globales válidas y consistentes, listas para ser utilizadas por la cabeza de regresión del modelo.

In [44]:
pool_mean = TemporalPooling("mean")
pool_last = TemporalPooling("last")

device = "cuda" if torch.cuda.is_available() else "cpu"

for fold in k_folds:
    print(f"\n=== Fold {fold} ===")

    # 1) Cargar ventanas 3D del fold desde diccionarios
    Xtr_k = Xtr[fold]
    Xva_k = Xva[fold]
    Xte_k = Xte[fold]

    # 2) Mini-batches
    xb_tr = torch.tensor(Xtr_k[:64], dtype=torch.float32).to(device)
    xb_va = torch.tensor(Xva_k[:64], dtype=torch.float32).to(device)
    xb_te = torch.tensor(Xte_k[:64], dtype=torch.float32).to(device)

    # 3) Encoder del fold desde diccionario
    enc = encoders[fold]

    # 4) Lista de sets de este fold
    pairs = [
        (xb_tr, enc, f"Fold{fold}-train"),
        (xb_va, enc, f"Fold{fold}-valid"),
        (xb_te, enc, f"Fold{fold}-test"),
    ]

    # 5) Pasar por encoder + pooling
    for xb, encoder, tag in pairs:
        with torch.no_grad():
            z  = encoder(xb)    # (64, T, d_model)
            p1 = pool_mean(z)   # (64, d_model)
            p2 = pool_last(z)   # (64, d_model)
        print(f'Para {tag}\n\tPool mean:\t{p1.shape}\tPool last:\t{p2.shape}')



=== Fold 1 ===
Para Fold1-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold1-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold1-test
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])

=== Fold 2 ===
Para Fold2-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold2-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold2-test
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])

=== Fold 3 ===
Para Fold3-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold3-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold3-test
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])

=== Fold 4 ===
Para Fold4-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold4-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold4-test

#####**5.2.1. Conclusión e interpretación de resultados**

Los resultados obtenidos son **correctos y coherentes** con la arquitectura definida.

- En cada evaluación se utilizan **mini-batches de 64 ventanas** provenientes de los conjuntos *train*, *valid* y *test* de cada fold, por lo que el tamaño de batch es 64.
- El encoder transforma cada entrada en una secuencia de embeddings con forma: `(64, T, 128)` donde `T = 90` corresponde a la longitud de la ventana y `128` es la dimensión latente definida por `d_model`.


- A continuación, los métodos de pooling temporal:
    - `pool_mean(z)` colapsa la dimensión temporal mediante un **promedio**, produciendo un tensor de forma `(64, 128)`.
    - `pool_last(z)` selecciona el **último paso temporal**, generando igualmente un tensor `(64, 128)`.

- La consistencia de las dimensiones entre folds es esperable, ya que:
    - Todos los folds comparten la misma arquitectura (`d_model = 128`).
    - Se utiliza el mismo tamaño de batch para la verificación.

En conjunto, se ha verificado que:

  - Los datos re-formateados (`Xtr_k`, `Xva_k`, `Xte_k`) poseen la forma adecuada y pueden ingresar correctamente al encoder.
  - Los encoders asociados a cada fold están correctamente inicializados y operan sin errores.
  - El módulo `TemporalPooling` funciona según lo esperado y produce **embeddings 2D** con forma `(batch, d_model)`, listos para ser utilizados por la cabeza final del modelo (regresión o clasificación).

Estos resultados confirman que el pipeline **encoder + pooling** está correctamente integrado y preparado para avanzar hacia la definición y entrenamiento de la cabeza de salida (head regression).


### **5.3. Inspección numérica de embeddings tras el pooling temporal**

Este bloque de código se utiliza para **inspeccionar los valores numéricos reales** de los embeddings generados por el modelo, luego de aplicar el encoder y el pooling temporal.

- Se instancian los módulos de pooling (`mean` y `last`), aunque en este caso se utiliza únicamente `mean`.
- Se selecciona automáticamente el dispositivo de cómputo disponible (GPU o CPU).

- Para cada *fold*:
  1. Se cargan las ventanas 3D del conjunto de entrenamiento desde el diccionario `Xtr`.
  2. Se construye un mini-batch de 64 muestras y se convierte a tensor en el dispositivo.
  3. Se recupera el encoder correspondiente al fold.
  4. En modo inferencia (`torch.no_grad()`), se ejecuta:
     - El encoder, obteniendo embeddings temporales con forma `(64, T, 128)`.
     - El pooling medio, colapsando la dimensión temporal a `(64, 128)`.

- Finalmente, se imprimen los **primeros 10 valores numéricos** del embedding correspondiente a la **primera muestra del batch**.

El objetivo principal es verificar que el pipeline *encoder + pooling* no solo produce las dimensiones correctas, sino también **valores numéricos válidos y finitos**, confirmando que los embeddings contienen información continua utilizable por la cabeza de salida del modelo.

In [45]:
pool_mean = TemporalPooling("mean")
pool_last = TemporalPooling("last")

device = "cuda" if torch.cuda.is_available() else "cpu"

for fold in k_folds:

    print(f"\n=== Fold {fold} ===")

    # 1) Cargar ventanas 3D desde diccionario
    Xtr_k = Xtr[fold]

    # 2) Mini-batch
    xb = torch.tensor(Xtr_k[:64], dtype=torch.float32).to(device)

    # 3) Encoder del fold desde diccionario
    enc = encoders[fold]

    # 4) Ejecutar encoder + pooling
    with torch.no_grad():
        z = enc(xb)               # (64, T, 128)
        p_mean = pool_mean(z)     # (64, 128)

    # 5) Mostrar valores numéricos reales del embedding
    print("Primeros 10 valores del embedding del fold:")
    print(p_mean[0, :10].cpu().numpy())



=== Fold 1 ===
Primeros 10 valores del embedding del fold:
[-0.36652985 -1.037119    0.9734119  -0.3910253  -0.93102795  0.34768882
 -0.33396757 -0.30811518  0.5198336  -0.4340472 ]

=== Fold 2 ===
Primeros 10 valores del embedding del fold:
[-0.90826756 -0.10013971  0.12322839  0.24266078 -0.47881454  1.0353489
  0.21347779  1.332372    0.9281702   0.06304602]

=== Fold 3 ===
Primeros 10 valores del embedding del fold:
[-1.0998359  -0.60294807 -0.38906187  1.0814722  -1.5690304  -1.4552183
 -0.36055812  0.8831734  -0.40101352 -0.623309  ]

=== Fold 4 ===
Primeros 10 valores del embedding del fold:
[-0.23071003 -1.4186257  -1.6576235  -0.42501348  0.4029681  -0.1131086
  1.7149905   0.37509507  0.2804585   1.0428237 ]

=== Fold 5 ===
Primeros 10 valores del embedding del fold:
[ 0.47868127 -0.2771579   0.0767173   0.4072622   1.0503738  -0.537181
 -0.4290558   0.5296356   2.890412   -1.0204655 ]


#### **5.3.1. Conclusión de la inspección numérica de embeddings**

Los valores impresos para cada fold son **coherentes y esperables**.

- Se observan valores **reales y finitos** (no aparecen `NaN` ni `Inf`), lo que indica que el pipeline: `X (ventanas 3D) → encoder → pooling(mean) → embedding (128)` está funcionando correctamente a nivel numérico.

- Los embeddings presentan valores tanto positivos como negativos, con magnitudes moderadas (aprox. entre -2 y 2 en los ejemplos), lo cual es típico en representaciones latentes generadas por redes neuronales con normalización y dropout.

- Es normal que los primeros 10 valores varíen entre folds, ya que:
  - Cada fold utiliza datos distintos.
  - Cada encoder es una instancia independiente (pesos inicializados separadamente, salvo que se haya fijado una semilla global y un orden de ejecución idéntico).

Esta prueba confirma que el modelo no solo respeta las dimensiones esperadas, sino que también produce embeddings numéricamente válidos, listos para ser consumidos por la cabeza de regresión en las siguientes etapas.

### **5.4. Para observar el contenido de Train, Valid y Test separados**

Este bloque de código se utiliza para **observar y comparar el contenido numérico de los embeddings** generados a partir de los conjuntos *train*, *valid* y *test*, de manera separada, para cada fold.

- Se instancian los módulos de pooling temporal (`mean` y `last`), utilizándose en este caso únicamente el **mean pooling**.
- Se selecciona automáticamente el dispositivo de cómputo disponible (GPU o CPU).

- Para cada *fold*:
  - Se cargan las ventanas 3D correspondientes a los conjuntos *train*, *valid* y *test* desde los diccionarios `Xtr`, `Xva` y `Xte`.
  - Se recupera el encoder asociado a ese fold, garantizando coherencia entre datos y modelo.

- Para cada conjunto (*train*, *valid*, *test*):
  - Se toma un mini-batch de 64 muestras.
  - Se ejecuta el pipeline **encoder + pooling** en modo inferencia (`torch.no_grad()`).
  - Se imprimen los **primeros 10 valores del embedding** correspondiente a la primera muestra del batch.

El objetivo principal es permitir una inspección cualitativa de los embeddings generados a partir de cada conjunto de datos, verificando que:
- El encoder y el pooling producen valores numéricos válidos en todos los sets.
- No existen anomalías evidentes entre *train*, *valid* y *test* antes de avanzar al entrenamiento completo del modelo.

In [46]:
pool_mean = TemporalPooling("mean")
pool_last = TemporalPooling("last")

device = "cuda" if torch.cuda.is_available() else "cpu"

for fold in k_folds:

    print(f"\n=== FOLD {fold} ===")

    # Cargar ventanas desde diccionarios
    Xtr_k = Xtr[fold]
    Xva_k = Xva[fold]
    Xte_k = Xte[fold]

    sets = {
        "train": Xtr_k,
        "valid": Xva_k,
        "test":  Xte_k
    }

    # Encoder desde diccionario
    enc = encoders[fold]

    for name, X in sets.items():

        xb = torch.tensor(X[:64], dtype=torch.float32).to(device)

        with torch.no_grad():
            z = enc(xb)
            p_mean = pool_mean(z)

        print(f"\n{name.upper()} — primeros 10 valores:")
        print(p_mean[0, :10].cpu().numpy())



=== FOLD 1 ===

TRAIN — primeros 10 valores:
[-0.38883898 -0.9818572   0.89213604 -0.30494645 -0.9075499   0.33045405
 -0.3896116  -0.35556558  0.5167616  -0.39605382]

VALID — primeros 10 valores:
[ 0.48515907 -0.30185637  0.15930557 -0.02095569  0.20145942  0.02336386
 -0.09857187 -0.7622383  -1.7117954   0.16426244]

TEST — primeros 10 valores:
[ 1.0709417   0.55539    -1.4409252  -0.5397553   1.410864    1.2879643
  0.22000092  0.30542663 -1.0142208   0.3922925 ]

=== FOLD 2 ===

TRAIN — primeros 10 valores:
[-0.7907213  -0.13680589 -0.00654467  0.15545428 -0.5151104   1.0497609
  0.18615055  1.3373712   0.81996226  0.08915574]

VALID — primeros 10 valores:
[-0.7630195   0.73850864 -0.16376558  0.28959897 -0.34571856  1.7886752
  0.39731023  0.07392097 -0.51971257  1.081344  ]

TEST — primeros 10 valores:
[-1.4049882   1.7763234  -0.16156031 -0.61099166  0.22355033  1.2051367
  0.3376715  -0.6439424  -1.3269235   1.5332484 ]

=== FOLD 3 ===

TRAIN — primeros 10 valores:
[-1.169325

####**5.4.1. Revisión del análisis y conclusión**

Los resultados confirman que el pipeline **fold → encoder → pooling** funciona correctamente. Además:

- Los embeddings difieren entre *train*, *valid* y *test* dentro de cada fold, y también entre folds, lo cual es esperable debido a la segmentación temporal y a la independencia de los encoders.

- No se observan colapsos, valores constantes ni anomalías numéricas (`NaN`/`Inf`).  
- El pooling genera representaciones 2D coherentes, listas para ser utilizadas por la cabeza de salida.

En síntesis, la generación de embeddings es estable y el flujo de datos por fold está correctamente implementado para avanzar al entrenamiento y tuneo del modelo.

##**6. Cabeza de regresión ('Regression Head') - Salida escalar**

La **cabeza de regresión** es el último bloque del modelo y cumple la función de **convertir el embedding generado por el encoder (y el pooling)** en una **predicción continua escalar**.


Conceptualmente:

- Recibe como entrada un embedding con forma: `(B, D)`
- Produce como salida un único valor por muestra: `(B,)`

Ese valor escalar puede representar, según el objetivo definido:
- El **retorno futuro**.
- La **dirección del precio** (si se modela como regresión continua).
- La **magnitud del movimiento**.
- Cualquier otra variable continua de interés.

### **6.1. Implementación de la cabeza de regresión**

La clase `RegressionHead` implementa una **red neuronal totalmente conectada (MLP)** simple y estable, diseñada para transformar el embedding en una predicción final.


**Arquitectura:**

- **Entrada**: embedding de dimensión `d_model` (por ejemplo, 128).
- **Primera capa lineal**: Reduce la dimensión del embedding de `D → D/2`.
- **Activación GELU**: Activación suave y estándar en arquitecturas Transformer.
- **Dropout**: Regularización para reducir el riesgo de *overfitting*.
- **Segunda capa lineal**: Proyecta de `D/2 → 1`, produciendo la salida escalar.

**Flujo dimensional:**

  `(B, D) → (B, D/2) → (B, 1) → (B,)`

  En el método `forward`, se aplica `squeeze(-1)` para eliminar la dimensión final innecesaria y devolver directamente un tensor 1D por batch.

El objetivo principal es transformar la representación latente aprendida por el encoder en una **predicción numérica directa**, cerrando el pipeline completo: `Ventanas → Encoder → Pooling → Regression Head → Predicción`.

Este diseño mantiene la arquitectura modular, permitiendo reutilizar el mismo encoder y pooling con distintas cabezas de salida si se desea cambiar el objetivo del modelo.



In [47]:
baseline_params

{'dropout': 0.1,
 'd_model': 128,
 'n_layers': 2,
 'n_heads': 8,
 'ff_mult': 2,
 'activation': 'gelu',
 'pooling': 'mean',
 'head_dropout': 0.1,
 'max_epochs': 50,
 'patience': 8,
 'lr': 0.0003,
 'weight_decay': 0.0001,
 'grad_clip': 1.0}

In [48]:
class RegressionHead(nn.Module):
    """
    Cabeza de regresión para modelos de series temporales.
    Toma un embedding de dimensión D (por ejemplo, 128) y produce
    un único valor escalar por muestra (predicción continua).
    """

    def __init__(self, d_model: int = 128, dropout: float = 0.1, activation: str = "relu"):
        super().__init__()

        act = self._get_activation(activation)

        # Red neuronal totalmente conectada (MLP) en dos capas:
        # 1) Proyección D -> D/2 con activación GELU.
        # 2) Proyección D/2 -> 1 (salida escalar).
        self.net = nn.Sequential(

            # Primera capa lineal: reduce la dimensión del embedding.
            # Entrada: (B, d_model)
            # Salida:  (B, d_model // 2)
            nn.Linear(d_model, d_model // 2),

            # GELU: activación usada en Transformers, suave y estable.
            nn.GELU(),

            # Dropout: regularización para evitar overfitting
            nn.Dropout(dropout),

            # Segunda capa lineal: produce un solo valor por muestra.
            # Entrada: (B, d_model // 2)
            # Salida:  (B, 1)
            nn.Linear(d_model // 2, 1)
        )

    @staticmethod
    def _get_activation(name: str) -> nn.Module:
        name = name.lower()
        if name == "relu":
            return nn.ReLU()
        if name == "gelu":
            return nn.GELU()
        if name == "tanh":
            return nn.Tanh()
        if name == "silu" or name == "swish":
            return nn.SiLU()
        raise ValueError(f"activation inválida: {name}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass de la cabeza de regresión.

        Parámetros
        ----------
        x : Tensor con forma (B, D)
            D es la dimensión del embedding producido por el encoder.

        Retorna
        -------
        Tensor con forma (B,)
            Un valor escalar predicho por cada muestra del batch.
        """

        # La red produce un tensor de forma (B, 1).
        # squeeze(-1) elimina la última dimensión para dejarlo en (B,).
        return self.net(x).squeeze(-1)

#### **6.1.1. Validación de la `RegressionHead` para la tarea MNQ**


La cabeza de regresión implementada es adecuada para nuestro problema, porque:

- El objetivo del modelo es un **valor escalar continuo**: el **retorno futuro a 90 minutos**.
- Luego del encoder + pooling obtenemos un embedding por muestra con forma: `(B, d_model) = (batch, 128)`
- Necesitamos una función que transforme ese embedding en una **predicción escalar**: `(B, 128) → (B,)`

La arquitectura propuesta (MLP simple con `Linear → GELU → Dropout → Linear`) es un enfoque **estándar y efectivo** para convertir embeddings de Transformers en salidas de regresión en tareas de forecasting, manteniendo buena capacidad de modelado y regularización.


### **6.2. Creación de la cabeza de regresión por fold**

En este bloque se instancian las **cabezas de regresión de forma independiente para cada fold**, manteniendo coherencia con el esquema de validación temporal.

- Se detecta automáticamente el dispositivo de cómputo disponible (GPU o CPU).
- Se crea un diccionario `heads` donde cada fold posee su **propia instancia** de `RegressionHead`.

Para cada fold:
- La cabeza se inicializa con:
  - `d_model = 128`, consistente con la dimensión del embedding producido por el encoder.
  - `dropout = 0.1`, como regularización.
- La instancia se traslada explícitamente al dispositivo definido.

El objetivo principal es asegurar que cada fold cuente con una cabeza de regresión independiente, evitando el uso compartido de parámetros entre folds y garantizando un entrenamiento y evaluación correctamente aislados.

In [49]:
from pandas.core.arrays import base
device = "cuda" if torch.cuda.is_available() else "cpu"

heads = {}   # Diccionario de heads por fold

for k in k_folds:
    heads[k] = RegressionHead(
        d_model=baseline_params['d_model'], #Hiperparametro tuneado
        dropout=baseline_params['head_dropout'],
        activation=baseline_params["activation"]#Hiperparametro tuneado
    ).to(device)

    print(f"Head creado para fold {k}")

Head creado para fold 1
Head creado para fold 2
Head creado para fold 3
Head creado para fold 4
Head creado para fold 5


### **6.3. Sanity check end-to-end (sin entrenar, solo shapes y un MSE “dummy”)**

Con el sanity check vamos a probar rápidamente que todo el pipeline funciona de punta a punta antes de entrenar.

El pipeline completo es:

`ventanas → encoder → pooling → cabeza de regresión → predicción escalar`

Se verifica que:
- Las ventanas ingresen al encoder y produzcan `z` con forma `(B, T, 128)`.
- El pooling reduzca la dimensión temporal y produzca `h` con forma `(B, 128)`.
- La cabeza de regresión entregue `yhat` con forma `(B,)`.
- No existan errores de shapes ni inconsistencias de batch entre las salidas.

Este chequeo **no entrena**, solo confirma que la arquitectura y los componentes por fold están integrados correctamente.


#### **6.3.1. Código de Sanity Check**

La función `sanity_check_pipeline(...)` recorre cada fold y, para cada conjunto (*train*, *valid*, *test*):

1. Recupera desde diccionarios:
   - `Xtr_k`, `Xva_k`, `Xte_k`
   - `encoders[fold]` y `heads[fold]`

2. Toma un mini-batch de tamaño `batch_size` (por defecto 64).

3. Ejecuta el flujo completo (en modo inferencia, sin gradientes):
   - `z = enc(xb)`  
   - `h = pool(z)`  
   - `yhat = head(h)`

4. Comprueba que las formas sean las esperadas:
   - `z` sea 3D
   - `h` sea 2D
   - `yhat` sea 1D
   - El tamaño de batch sea consistente entre `xb`, `h` y `yhat`

5. Imprime por set:
   - Las shapes si todo está correcto, o un mensaje de error si no lo está.
   - Un resumen por fold indicando si el pipeline completo está OK o si hubo problemas.

El objetivo principal es confirmar que el modelo está bien cableado de punta a punta en cada fold, antes de avanzar al entrenamiento y al tuneo de hiperparámetros.

In [50]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Pooling temporal
pool = TemporalPooling(baseline_params['pooling']).to(device)

def sanity_check_pipeline(k_folds, Xtr, Xva, Xte, encoders, heads, batch_size=64):
    """
    Verifica el pipeline completo encoder → pooling → head de regresión
    para cada fold, usando mini-batches chicos.
    """

    for fold in k_folds:
        print(f"\n=== Sanity check FOLD {fold} ===")

        # 1) Recuperar estructuras desde diccionarios
        try:
            Xtr_k = Xtr[fold]
            Xva_k = Xva[fold]
            Xte_k = Xte[fold]

            enc  = encoders[fold]
            head = heads[fold]

        except KeyError as e:
            print(f"  Faltan datos o modelos para el fold {fold}: {e}")
            continue

        sets = {
            "train": Xtr_k,
            "valid": Xva_k,
            "test":  Xte_k
        }

        fold_ok = True

        for nombre_set, X in sets.items():

            if X is None or len(X) == 0:
                print(f"  {nombre_set}: sin datos, se omite.")
                continue

            # 2) Mini-batch chico
            xb = torch.tensor(X[:batch_size], dtype=torch.float32).to(device)

            with torch.no_grad():
                # Paso 1: encoder
                z = enc(xb)          # (B, T, D)

                # Paso 2: pooling
                h = pool(z)          # (B, D)

                # Paso 3: head de regresión
                yhat = head(h)       # (B,)

            # 3) Comprobación de shapes
            ok_shapes = (
                z.ndim == 3 and
                h.ndim == 2 and
                yhat.ndim == 1 and
                xb.shape[0] == h.shape[0] == yhat.shape[0]
            )

            if ok_shapes:
                print(f"  {nombre_set}: z{tuple(z.shape)} → h{tuple(h.shape)} → yhat{tuple(yhat.shape)}")
            else:
                print(f"  {nombre_set}: SHAPES ERROR: "
                      f"z{tuple(z.shape)}, h{tuple(h.shape)}, yhat{tuple(yhat.shape)}")
                fold_ok = False

        if fold_ok:
            print(f"  ✔️ Pipeline COMPLETO OK en FOLD {fold}.")
        else:
            print(f"  ❌ Problemas de shapes en FOLD {fold}.")


In [51]:
sanity_check_pipeline(k_folds, Xtr, Xva, Xte, encoders, heads)


=== Sanity check FOLD 1 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
  ✔️ Pipeline COMPLETO OK en FOLD 1.

=== Sanity check FOLD 2 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
  ✔️ Pipeline COMPLETO OK en FOLD 2.

=== Sanity check FOLD 3 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
  ✔️ Pipeline COMPLETO OK en FOLD 3.

=== Sanity check FOLD 4 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
  ✔️ Pipeline COMPLETO OK en FOLD 4.

=== Sanity check FOLD 5 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → 

#### **6.3.2. Conclusión del sanity check end-to-end**

Los resultados confirman que el **pipeline completo del modelo funciona correctamente en todos los folds**.

Para cada fold y para los conjuntos *train*, *valid* y *test* se verifica que:
- El encoder produce salidas con forma `(64, 90, 128)`.
- El pooling reduce correctamente la dimensión temporal a `(64, 128)`.
- La cabeza de regresión genera predicciones escalares con forma `(64,)`.

No se detectan errores de *shape*, inconsistencias de batch ni problemas de integración entre módulos.

En síntesis, la arquitectura **ventanas → encoder → pooling → RegressionHead → predicción escalar** está correctamente conectada y validada, y el modelo se encuentra listo para iniciar la etapa de entrenamiento y tuneo de hiperparámetros.

Antes de entrenar, es fundamental asegurarnos de que:

- Las ventanas están bien formateadas (3D correctas).  
- El encoder procesa correctamente la secuencia.  
- El pooling reduce correctamente la dimensión temporal.  
- La cabeza de regresión produce un escalar por muestra.  
- Todo funciona en CPU o GPU sin errores.

Este paso nos garantiza que el pipeline entero está sano y listo para el entrenamiento real.

### **6.4. Sanity check de pérdida (MSE).**

Este bloque de código se utiliza para **generar predicciones escalares (`ŷ`)** a partir del pipeline completo del modelo, **sin realizar entrenamiento**, únicamente para verificar el flujo end-to-end y preparar estructuras de salida.

- Se define el dispositivo de cómputo (CPU o GPU) y un `batch_size` fijo de 64.
- Se instancia el módulo de **pooling temporal** (`mean`) en el dispositivo.
- Se inicializan diccionarios para almacenar las predicciones de *train*, *valid* y *test* por fold.




#### **6.4.1. Generación de ŷ por fold (inferencia sin entrenamiento)**



Para cada *fold*:

1. **Carga de datos y modelos**  
   - Se recuperan las ventanas 3D (`Xtr_k`, `Xva_k`, `Xte_k`).
   - Se obtienen el `encoder` y la `RegressionHead` correspondientes al fold.

2. **Selección de mini-batches**  
   - Se toman las primeras `batch_size` muestras de cada conjunto.
   - Se convierten a tensores y se envían al dispositivo.

3. **Ejecución del pipeline completo (sin gradientes)**  
   Para *train*, *valid* y *test*: `ventanas → encoder → pooling → RegressionHead → ŷ`

    - El encoder produce `(B, T, D)`.
    - El pooling reduce a `(B, D)`.
    - La cabeza de regresión genera predicciones `(B,)`.

4. **Almacenamiento de predicciones**  
    - Las predicciones se pasan a CPU y se guardan como `numpy arrays` en diccionarios:
      - `yhat_train[fold]`
      - `yhat_valid[fold]`
      - `yhat_test[fold]`

5. **Verificación de shapes**  
    - Se imprimen las dimensiones de cada `ŷ` para confirmar coherencia.

El objetivo principal es:

- Confirmar que el modelo produce **predicciones escalares válidas** para cada conjunto y fold.
- Verificar que el pipeline completo funciona sin errores numéricos o de *shape*.
- Preparar estructuras de salida (`ŷ`) que luego podrán compararse con los valores reales (`y`) o utilizarse en métricas, todo **antes del entrenamiento real**.

In [52]:
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 64

# Pooling temporal
pool = TemporalPooling(baseline_params['pooling']).to(device)

# Diccionarios para guardar las predicciones
yhat_train = {}
yhat_valid = {}
yhat_test  = {}

for fold in k_folds:
    print(f"\n=== Generando yhat para Fold {fold} ===")

    # 1) Datos del fold desde diccionarios
    Xtr_k = Xtr[fold]
    Xva_k = Xva[fold]
    Xte_k = Xte[fold]

    enc  = encoders[fold].to(device)
    head = heads[fold].to(device)

    # 2) Mini‐batches (solo las primeras batch_size muestras)
    xb_tr = torch.tensor(Xtr_k[:batch_size], dtype=torch.float32).to(device)
    xb_va = torch.tensor(Xva_k[:batch_size], dtype=torch.float32).to(device)
    xb_te = torch.tensor(Xte_k[:batch_size], dtype=torch.float32).to(device)

    with torch.no_grad():
        # ---- TRAIN ----
        z_tr    = enc(xb_tr)          # (B, T, D)
        h_tr    = pool(z_tr)          # (B, D)
        yhat_tr = head(h_tr)          # (B,)
        yhat_train[fold] = yhat_tr.cpu().numpy()

        # ---- VALID ----
        z_va    = enc(xb_va)
        h_va    = pool(z_va)
        yhat_va = head(h_va)
        yhat_valid[fold] = yhat_va.cpu().numpy()

        # ---- TEST ----
        z_te    = enc(xb_te)
        h_te    = pool(z_te)
        yhat_te = head(h_te)
        yhat_test[fold] = yhat_te.cpu().numpy()

    print(f"  yhat_train[{fold}].shape =", yhat_train[fold].shape)
    print(f"  yhat_valid[{fold}].shape =", yhat_valid[fold].shape)
    print(f"  yhat_test[{fold}].shape  =", yhat_test[fold].shape)



=== Generando yhat para Fold 1 ===
  yhat_train[1].shape = (64,)
  yhat_valid[1].shape = (64,)
  yhat_test[1].shape  = (64,)

=== Generando yhat para Fold 2 ===
  yhat_train[2].shape = (64,)
  yhat_valid[2].shape = (64,)
  yhat_test[2].shape  = (64,)

=== Generando yhat para Fold 3 ===
  yhat_train[3].shape = (64,)
  yhat_valid[3].shape = (64,)
  yhat_test[3].shape  = (64,)

=== Generando yhat para Fold 4 ===
  yhat_train[4].shape = (64,)
  yhat_valid[4].shape = (64,)
  yhat_test[4].shape  = (64,)

=== Generando yhat para Fold 5 ===
  yhat_train[5].shape = (64,)
  yhat_valid[5].shape = (64,)
  yhat_test[5].shape  = (64,)


#### **6.4.2. Sanity check de MSE por fold (predicciones vs targets escalados)**

Este bloque realiza un **chequeo numérico básico** comparando las predicciones generadas (`ŷ`) con los valores reales escalados (`y`) para cada fold, sin entrenar el modelo.

Para cada fold:

1. **Carga los targets reales** (*train*, *valid* y *test*) desde:
   - `y_train_sc`, `y_valid_sc`, `y_test_sc`.

2. **Recupera las predicciones previamente generadas**:
   - `yhat_train`, `yhat_valid`, `yhat_test`.

3. **Selecciona un mini-batch** de tamaño `batch_size` (por defecto 64) para cada set.

4. **Convierte a tensores** las predicciones y los valores reales, asegurando:
   - Tipo `float32`.
   - Forma 1D `(B,)`.
   - Dispositivo consistente (CPU/GPU).

5. **Calcula el error cuadrático medio (MSE)** entre `ŷ` y `y`: `MSE(ŷ, y)`

6. **Imprime por set**:
    - La forma de las predicciones.
    - El valor numérico del MSE.


El objetivo principal es verificar que:
- Las predicciones y los targets tienen **formas compatibles**.
- El cálculo de la función de pérdida funciona correctamente.
- No existen errores numéricos (`NaN`, `Inf`) ni inconsistencias de dispositivo.

Este sanity check **no evalúa performance**, solo confirma que el modelo puede calcular una pérdida válida antes de iniciar el entrenamiento real.


In [53]:
def sanity_check_mse_folds(k_folds, y_train_sc, y_valid_sc, y_test_sc,
                           yhat_train, yhat_valid, yhat_test,
                           batch_size=64):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    for fold in k_folds:
        print(f"\n=== Sanity check MSE — Fold {fold} ===")

        # 1) Targets del fold desde diccionarios
        ytr = y_train_sc[fold]
        yva = y_valid_sc[fold]
        yte = y_test_sc[fold]

        # 2) Predicciones generadas anteriormente
        yhat_tr = yhat_train[fold]
        yhat_va = yhat_valid[fold]
        yhat_te = yhat_test[fold]

        sets = [
            ("train", ytr[:batch_size], yhat_tr[:batch_size]),
            ("valid", yva[:batch_size], yhat_va[:batch_size]),
            ("test",  yte[:batch_size], yhat_te[:batch_size]),
        ]

        for name, y_true, y_pred in sets:

            # Convertir a tensores
            yb = torch.tensor(y_true, dtype=torch.float32, device=device).view(-1)
            yh = torch.tensor(y_pred, dtype=torch.float32, device=device).view(-1)

            # Calcular MSE
            loss = torch.nn.functional.mse_loss(yh, yb)

            print(f"  {name:<6} — yhat:{tuple(yh.shape)}  MSE={float(loss):.6f}")


In [54]:
sanity_check_mse_folds(
    k_folds,
    y_train_sc, y_valid_sc, y_test_sc,
    yhat_train, yhat_valid, yhat_test
)


=== Sanity check MSE — Fold 1 ===
  train  — yhat:(64,)  MSE=0.050139
  valid  — yhat:(64,)  MSE=0.004779
  test   — yhat:(64,)  MSE=0.015310

=== Sanity check MSE — Fold 2 ===
  train  — yhat:(64,)  MSE=0.005496
  valid  — yhat:(64,)  MSE=0.003489
  test   — yhat:(64,)  MSE=0.026696

=== Sanity check MSE — Fold 3 ===
  train  — yhat:(64,)  MSE=0.050213
  valid  — yhat:(64,)  MSE=0.017995
  test   — yhat:(64,)  MSE=0.109907

=== Sanity check MSE — Fold 4 ===
  train  — yhat:(64,)  MSE=0.006188
  valid  — yhat:(64,)  MSE=0.032116
  test   — yhat:(64,)  MSE=0.050364

=== Sanity check MSE — Fold 5 ===
  train  — yhat:(64,)  MSE=0.044079
  valid  — yhat:(64,)  MSE=0.004397
  test   — yhat:(64,)  MSE=0.016123


### **6.4.3. Conclusiones del sanity check de MSE por fold**

A partir de los valores obtenidos de MSE para cada fold, se pueden establecer las siguientes conclusiones:

1. El pipeline funciona correctamente en todos los folds

    - En todos los casos, las predicciones `yhat` presentan la forma esperada `(64,)`.
    - No se registraron errores de dimensiones, tipos de datos ni conflictos de dispositivo (CPU/GPU).
    - Esto confirma que el flujo completo `encoder → pooling → RegressionHead` opera sin inconsistencias técnicas.

2. Los valores de MSE son coherentes con un modelo no entrenado

    - Los pesos del encoder y de la cabeza de regresión **aún no han sido entrenados**, por lo que las predicciones son esencialmente aleatorias.
    - En este contexto:
    - Es esperable que el MSE varíe entre folds y entre conjuntos (*train*, *valid* y *test*).
    - El objetivo de este chequeo no es minimizar el error, sino verificar que el cálculo de la pérdida sea posible y estable.
    - Los valores observados reflejan este comportamiento, con MSE de distinta magnitud según el fold y el conjunto, lo cual es normal en esta etapa.

3. El modelo está listo para avanzar al entrenamiento real

    - El pipeline completo ha sido validado tanto en términos de **formas** como de **cálculo de la función de pérdida**.
    - Se verificó exitosamente que:
    - `ventanas → encoder → pooling → head → yhat`
    - `yhat` puede compararse con los targets reales mediante MSE sin errores.
    - El siguiente paso consiste en implementar el bucle de entrenamiento por fold, incorporando:
    - función de pérdida,
    - optimizador,
    - (opcionalmente) *learning rate scheduling*,
    - métricas de evaluación (RMSE, MAE, SMAPE, Directional Accuracy).

En síntesis, estos resultados confirman que el modelo se encuentra **estructuralmente sano** y preparado para iniciar el entrenamiento sin inconvenientes.

## **7. Preparación para Entrenamiento**

### **7.1. Dataset + DataLoader (reshape dentro)**

El siguiente apartado prepara todo lo necesario para entrenar un modelo en PyTorch usando nuestras ventanas:

1. Escala los valores objetivo (y) usando StandardScaler.
    - Esto ayuda a estabilizar el entrenamiento.
    - El scaler se ajusta solo con y_train (buena práctica).

2. Convierte tus ventanas X (aplanadas en 2D) a tensores 3D (B, T, F)
donde:
    - B = batch size
    - T = tamaño de la ventana temporal (90 minutos)
    - F = número de features

3. Construye un Dataset personalizado (WindowDataset)
    - Guarda X y y en formato listo para PyTorch.
    - Aplica el escalador únicamente a y.

4. Crea dataloaders para entrenamiento y validación
    - `dl_tr`: con shuffle=True
    - `dl_va`: sin shuffle, para evaluación estable
    - Ambos con pin_memory=True (optimiza transferencias CPU→GPU)

Este bloque no entrena nada todavía, pero prepara correctamente los datos para alimentar el modelo fold por fold.

In [55]:
import os, joblib
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

'''1) Scaler de y (fit solo con y_train)
 Propósito: Normalizar y para facilitar el entrenamiento y evitar escalas muy pequeñas o muy grandes.
'''
def get_y_scaler(y_train: np.ndarray, path: str = None):
    # Crea un StandardScaler y lo ajusta solo con y_train.
    scaler = StandardScaler()
    scaler.fit(y_train.reshape(-1, 1))   # y debe ser columna

    # Si se pasa un path, guarda el scaler en disco.
    if path:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        joblib.dump(scaler, path)

    return scaler
'''
2) Dataset que aplica el y_scaler
Propósito: PyTorch necesita un Dataset para entregar lotes de entrenamiento.
Aquí se reconstruyen las ventanas (T,F) y se devuelven como tensores.
'''
class WindowDataset(Dataset):
    def __init__(self, X_flat, y, T, F, y_scaler: StandardScaler):
        # Verifica que X_flat tenga la forma correcta: (N, T*F)
        assert X_flat.shape[1] == T * F, f"Inconsistente: {X_flat.shape[1]} != {T}*{F}"

        # Convierte ventana 2D a 3D: (N, T*F) → (N, T, F)
        X = X_flat.reshape(-1, T, F).astype(np.float32)

        # Si usamos scaler, transformamos y y lo convertimos a float32
        if y_scaler is not None:
            y = y_scaler.transform(y.reshape(-1, 1)).ravel()

        self.X = X
        self.y = y.astype(np.float32)

    def __len__(self):
        # Cantidad total de muestras
        return len(self.y)

    def __getitem__(self, i):
        # Devuelve la i-ésima ventana y su target como tensores PyTorch
        return torch.from_numpy(self.X[i]), torch.tensor(self.y[i], dtype=torch.float32)

'''
3) Loaders genéricos (cualquier horizonte)
Propósito: Generar los iteradores que el modelo usará durante el entrenamiento:
      - dl_tr: batches mezclados
      - dl_va: batches ordenados (evaluación estable)
'''

def make_loaders(
    Xtr, ytr, Xva, yva, T, F, y_scaler,
    bs=256,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
):
    ds_tr = WindowDataset(Xtr, ytr, T, F, y_scaler=y_scaler)
    ds_va = WindowDataset(Xva, yva, T, F, y_scaler=y_scaler)

    #persistent_workers solo tiene sentido si num_workers > 0
    persistent_workers = bool(persistent_workers and num_workers > 0)

    dl_tr = DataLoader(
        ds_tr,
        batch_size=bs,
        shuffle=True,
        pin_memory=pin_memory,
        num_workers=num_workers,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor if num_workers > 0 else None,
    )

    dl_va = DataLoader(
        ds_va,
        batch_size=bs,
        shuffle=False,
        pin_memory=pin_memory,
        num_workers=num_workers,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor if num_workers > 0 else None,
    )

    return dl_tr, dl_va


El bloque anterior construye el pipeline que convierte tus dataframes en: `Ventanas 3D → Dataset PyTorch → DataLoader → Entrenamiento`

Transforma:
 - X a (B, T, F)
 - y a valores escalados

### **7.2. Modelo compacto por fold (encoder + pooling mean + head)**

Un modelo compacto por fold: `modelo_k = encoder_k + pooling + head_k`

Un modelo compacto por fold combina las tres partes del pipeline (encoder → pooling → head) en un único `nn.Module`.  

Se decidió utilizar un modelo compacto por las siguientes razones:

1. Permite que **cada fold tenga un modelo completamente independiente**, evitando fuga de información entre folds.  
2. Simplifica el loop de entrenamiento: en lugar de ejecutar manualmente `encoder → pool → head`, el modelo produce directamente la predicción `ŷ = model(x)`.  
3. Facilita el uso de optimizadores, carga/guardado de pesos y evaluación, ya que todos los parámetros entrenables quedan dentro de un único módulo por fold.  
4. Mantiene una estructura clara: el “modelo” es la combinación natural de encoder, reducción temporal y cabeza de regresión.

Con esto, el punto 7.3 (loop de entrenamiento) puede trabajar con un único módulo (`model_k`) por fold, lo cual hace el código más limpio y menos propenso a errores.


In [56]:
print(pool)

TemporalPooling()


In [57]:
#assert pool.mode == "last", f"Pooling incorrecto: {pool.mode}"
#print("✔ TemporalPooling configurado en 'last'")

In [58]:
baseline_models = {}   # diccionario para almacenar modelos completos por fold

for fold in k_folds:

    encoder = encoders[fold]
    head    = heads[fold]

    # pooling es compartido
    model = nn.Sequential(
        encoder,   # (B, T, F) → (B, T, d_model)
        pool,      # (B, T, d_model) → (B, d_model)
        head       # (B, d_model) → (B,)
    )

    baseline_models[fold] = model



In [59]:
#Como acceder
#modelo_fold_3 = models[3]
#y_pred = modelo_fold_3(x_batch)

### **7.3. Loop de entrenamiento (MSE, AdamW, early stopping simple)**

El siguiente bloque implementa el loop de entrenamiento del modelo por fold.
Entrena un encoder + pooling + cabeza de regresión usando MSE como función de pérdida, optimizador AdamW, soporte opcional para AMP (mixed precision), clipping de gradiente y un esquema simple de early stopping basado en la pérdida de validación. El objetivo es obtener un modelo estable y con buena generalización, ajustando solo los parámetros del encoder y de la cabeza,mientras que el pooling permanece fijo.


In [60]:
windows_size = 90

In [61]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

def train_model(model: nn.Module,
                dl_tr: DataLoader,
                dl_va: DataLoader,
                device: str = "cuda" if torch.cuda.is_available() else "cpu",
                #HIPERPARAMETROS DE ENTRENAMIENTO
                #Learning Rate
                lr: float = 3e-4,
                #Regularización L2
                weight_decay: float = 1e-4,
                #Máxima cantidad de épocas
                max_epochs: int = 50,
                #Paciencia de Early Stopping
                patience: int = 8,
                #Clipping de gradiente
                grad_clip: float = 1.0,
                #Uso de Mixed Precision (True/False)
                use_amp: bool = True):
    """
    Entrena un modelo compacto (encoder + pooling + cabeza de regresión)
    usando MSE como función de pérdida, AdamW como optimizador y un esquema
    simple de early stopping basado en la pérdida de validación.

    Parámetros
    ----------
    model : nn.Module
        Modelo completo (por ejemplo: nn.Sequential(encoder, pool, head)).
    dl_tr : DataLoader
        DataLoader de entrenamiento.
    dl_va : DataLoader
        DataLoader de validación.
    device : str
        "cuda" si hay GPU disponible, de lo contrario "cpu".
    lr : float
        Learning rate del optimizador AdamW.
    weight_decay : float
        Término de regularización L2 (weight decay) de AdamW.
    max_epochs : int
        Máximo número de épocas de entrenamiento.
    patience : int
        Número de épocas sin mejora en validación antes de activar early stopping.
    grad_clip : float
        Valor máximo de norma de gradiente para aplicar gradient clipping.
        Si es None, no se aplica clipping.
    use_amp : bool
        Si es True y hay GPU, activa mixed precision (AMP) para acelerar el entrenamiento.

    Retorna
    -------
    model : nn.Module
        Modelo con los mejores pesos encontrados (según pérdida de validación).
    """

    # Enviar todo el modelo al dispositivo (GPU/CPU)
    model = model.to(device)

    # Optimizador AdamW (recomendado para arquitecturas tipo Transformer)
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # GradScaler para entrenamiento en mixed precision (solo en GPU)
    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device == "cuda"))

    # Variables para seguimiento del mejor modelo (early stopping)
    best_val = float("inf")   # mejor pérdida de validación observada
    best_state = None         # state_dict del mejor modelo
    noimp = 0                 # épocas consecutivas sin mejora

    # ==========================================================
    #                      LOOP DE ÉPOCAS
    # ==========================================================
    for epoch in range(1, max_epochs + 1):

        # ----------------------- ENTRENAMIENTO -----------------------
        model.train()
        tr_loss = 0.0

        for xb, yb in dl_tr:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            # Reset del gradiente
            opt.zero_grad(set_to_none=True)

            # Forward con AMP opcional
            with torch.cuda.amp.autocast(enabled=(use_amp and device == "cuda")):
                # El modelo compacto incluye: encoder → pool → head
                yhat = model(xb).view(-1)  # salida (B,)
                loss = nn.functional.mse_loss(yhat, yb)

            # Backpropagation con GradScaler
            scaler.scale(loss).backward()
            scaler.unscale_(opt)  # necesario antes del clipping

            # Clipping de gradiente para evitar explosiones
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            # Paso de optimización
            scaler.step(opt)
            scaler.update()

            # Acumulación de la pérdida ponderada por el tamaño del batch
            tr_loss += loss.item() * xb.size(0)

        # Promedio de pérdida de entrenamiento por muestra
        tr_loss /= len(dl_tr.dataset)

        # ----------------------- VALIDACIÓN -----------------------
        model.eval()
        va_loss = 0.0

        with torch.no_grad():
            for xb, yb in dl_va:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)


                yhat = model(xb).view(-1)
                va_loss += nn.functional.mse_loss(yhat, yb).item() * xb.size(0)

        # Promedio de pérdida de validación por muestra
        va_loss /= len(dl_va.dataset)

        # Log de la época
        print(f"Epoch {epoch:03d}  train={tr_loss:.6e}  valid={va_loss:.6e}")

        # ----------------------- EARLY STOPPING -----------------------
        if va_loss < best_val - 1e-9:
            # Mejora en validación: se guarda el mejor modelo hasta ahora
            best_val = va_loss
            noimp = 0
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            # No hubo mejora: se incrementa el contador
            noimp += 1
            if noimp >= patience:
                print("Early stopping por falta de mejora en validación.")
                break

    # Restaurar los mejores pesos encontrados
    if best_state is not None:
        model.load_state_dict(best_state)

    return model


In [62]:
## Para ejecutar el código

'''
for fold in k_folds:
    model_k = globals()[f"model_{fold}"]
    dl_tr_k, dl_va_k = ...  # loaders del fold k
    print(f"\n=== Entrenando modelo del Fold {fold} ===")
    model_k = train_model(model_k, dl_tr_k, dl_va_k)
    globals()[f"model_{fold}"] = model_k
'''

'\nfor fold in k_folds:\n    model_k = globals()[f"model_{fold}"]\n    dl_tr_k, dl_va_k = ...  # loaders del fold k\n    print(f"\n=== Entrenando modelo del Fold {fold} ===")\n    model_k = train_model(model_k, dl_tr_k, dl_va_k)\n    globals()[f"model_{fold}"] = model_k\n'

### **7.4. Inferencia (Predicción)**

La siguiente función realiza inferencia (predicción) en un conjunto completo de ventanas sin calcular gradientes, usando el pipeline: `encoder → pooling → head → predicción escalar`

Sirve para obtener todas las predicciones de train, valid o test después de entrenar el modelo por fold.

En detalle:

1. Convierte X_flat (que viene en formato (N, T*F)) a ventanas 3D (N, T, F)
2. Pasa por el modelo en batches grandes (4096 por defecto) para acelerar la inferencia
3. Obtiene las predicciones ŷ
4. Si las predicciones están escaladas, aplica inverse_transform del scaler de y
5. Devuelve un array 1D con las predicciones reales

Es decir: **Esta función transforma un dataset completo en sus predicciones finales del modelo.**

Se usa después de entrenar, tipicamente para:
  - Evaluar rendimiento
  - Graficar pred vs real
  - Guardar resultados
  - Calcular RMSE, MAE, SMAPE, DA, etc.

In [63]:
@torch.no_grad()
def predict_set(enc, pool, head, X_flat, T, F, device, batch_size=4096, y_scaler=None):
    N = X_flat.shape[0]
    preds = []

    enc.eval(); head.eval()

    for i in range(0, N, batch_size):
        xb_np = X_flat[i:i+batch_size].reshape(-1, T, F).astype(np.float32, copy=False)
        xb = torch.from_numpy(xb_np).to(device, non_blocking=True)

        z = enc(xb)
        h = pool(z)
        yb = head(h).detach().cpu().numpy()
        preds.append(yb)

        # libera referencia del batch (ayuda al GC)
        del xb, z, h

    y_pred_scaled = np.concatenate(preds, axis=0).reshape(-1, 1)
    return (y_scaler.inverse_transform(y_pred_scaled).ravel() if y_scaler is not None else y_pred_scaled.ravel())


En resumen:
- Esta función realiza predicción vectorizada, sin gradientes.
- Usa el pipeline completo: encoder → pooling → head.
- Procesa el dataset en batches grandes (eficiente).
- Reconstruye ventanas desde 2D → 3D.
- Aplica inverse_transform del scaler de y si corresponde.
- Devuelve un vector plano con todas las predicciones del modelo.

### **7.5. Rutas para guardar modelos por fold**

In [64]:
import os

# --- Función unificada ---
def ruta_modelo_fold(fold: int, subcarpeta: str) -> dict:
    """
    Crea la carpeta destino y devuelve la ruta completa
    para almacenar el modelo correspondiente al fold.
    """
    base = f"{drive_path}/5_transformer_model/{subcarpeta}"
    os.makedirs(base, exist_ok=True)

    model_path = os.path.join(base, f"baseline_fold_{fold}.pt")
    return {"model_path": model_path}



In [65]:
# --- Generar diccionario de rutas ---
subcarpeta = "5_3_baseline_model"
path_models = {}

for k in k_folds:
    path_models[k] = ruta_modelo_fold(k, subcarpeta)

# --- Resultado final ---
path_models

{1: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_model/5_3_baseline_model/baseline_fold_1.pt'},
 2: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_model/5_3_baseline_model/baseline_fold_2.pt'},
 3: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_model/5_3_baseline_model/baseline_fold_3.pt'},
 4: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_model/5_3_baseline_model/baseline_fold_4.pt'},
 5: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_model/5_3_baseline_model/baseline_fold_5.pt'}}

# Entrenamiento Transformers (Baseline)

## **8. Entrenamiento**

In [66]:
device = "cuda" if torch.cuda.is_available() else "cpu"

### **8.1. Entrenamiento de heavy_models**

In [67]:
#heavy_train_params

In [68]:
flag_baseline_folds_metrics

False

In [69]:
models    = {}
scalers_y  = {}
ytr_pred_d = {}
yva_pred_d = {}
yte_pred_d = {}

In [73]:
device = "cuda" if torch.cuda.is_available() else "cpu"
  # Tamaño de ventana y cantidad de features
T = windows_size # longitud de la ventana temporal
F = len(features_90) # número de features por paso
batch_train = 11246 # batch size para entrenamiento
batch_pred = 8192
# Pooling compartido (sin parámetros entrenables)
pool = TemporalPooling(baseline_params['pooling']).to(device)

# ------------------------------------------------------------
# Control global: entrenar o no según metrics_k_folds
# ------------------------------------------------------------
if flag_baseline_folds_metrics:
      print("flag_baseline_folds_metrics=True → se omite el entrenamiento de todos los folds.")
else:
      print("flag_baseline_metrics=False → se inicia entrenamiento por folds.")

      for fold in k_folds:
          #Definir una clave de modelo por fold (para registro de métricas)
          model_key = f"baseline_fold_{fold}"

          # Si ya existe en la tabla de métricas, omitimos SOLO ese fold
          if ("baseline_folds_metrics" in globals()
              and baseline_folds_metrics is not None
              and model_key in baseline_folds_metrics.index):
              print(f"Omitimos este entrenamiento: {model_key} ya existe en baseline_folds_metrics")
              continue

          print(f"\n=== Entrenando modelo: {model_key} ===")

          # ------------------------------------------------------------
          # 2) Recuperar X e y del fold
          # ------------------------------------------------------------
          Xtr_3d = Xtr[fold]
          Xva_3d = Xva[fold]
          Xte_3d = Xte[fold]

          ytr = y_train_sc[fold]
          yva = y_valid_sc[fold]
          yte = y_test_sc[fold]

          # Aplanar 3D -> 2D
          Xtr_flat = Xtr_3d.reshape(Xtr_3d.shape[0], -1)
          Xva_flat = Xva_3d.reshape(Xva_3d.shape[0], -1)
          Xte_flat = Xte_3d.reshape(Xte_3d.shape[0], -1)

          # ------------------------------------------------------------
          # 3) Scaler de y (solo train)
          # ------------------------------------------------------------
          scaler_y_fold = get_y_scaler(ytr)

          # ------------------------------------------------------------
          # 4) DataLoaders
          # ------------------------------------------------------------
          dl_tr_fold, dl_va_fold = make_loaders(
              Xtr_flat, ytr,
              Xva_flat, yva,
              T=T, F=F,
              y_scaler=scaler_y_fold,
              bs=batch_train,
              num_workers=8,
              prefetch_factor=4,
              pin_memory=True,
              persistent_workers=False
          )

          # ------------------------------------------------------------
          # 5) Modelo del fold
          # ------------------------------------------------------------
          encoder_fold = encoders[fold].to(device)
          head_fold    = heads[fold].to(device)

          model_fold = nn.Sequential(
              encoder_fold,
              pool,
              head_fold
          ).to(device)

          # ------------------------------------------------------------
          # 6) Entrenamiento
          # ------------------------------------------------------------
          model_fold = train_model(
              model_fold,
              dl_tr_fold,
              dl_va_fold,
              device=device,
              max_epochs=baseline_params['max_epochs'],
              patience=baseline_params['patience'],
              use_amp=True,
              lr=baseline_params['lr'],                       #lr=3e-4,
              weight_decay = baseline_params['weight_decay'], #weight_decay=1e-4,
              grad_clip=baseline_params['grad_clip'],         #grad_clip=1.0,
          )

          # ------------------------------------------------------------
          # 7) Predicciones
          # ------------------------------------------------------------
          ytr_pred = predict_set(
              encoder_fold, pool, head_fold,
              Xtr_flat, T, F,
              device,
              batch_size=batch_pred,
              y_scaler=scaler_y_fold
          )

          yva_pred = predict_set(
              encoder_fold, pool, head_fold,
              Xva_flat, T, F,
              device,
              batch_size=batch_pred,
              y_scaler=scaler_y_fold
          )

          yte_pred = predict_set(
              encoder_fold, pool, head_fold,
              Xte_flat, T, F,
              device,
              batch_size=batch_pred,
              y_scaler=scaler_y_fold
          )

          # ------------------------------------------------------------
          # 8) Cálculo de métricas por fold (train / valid / test)
          #    Usamos evaluate_model, pasando y_pred explícitamente.
          # ------------------------------------------------------------
          metrics_tr = evaluate_model(None, None, ytr, ytr_pred)
          metrics_va = evaluate_model(None, None, yva, yva_pred)
          metrics_te = evaluate_model(None, None, yte, yte_pred)

          # ------------------------------------------------------------
          # 9) Guardar métricas en el DataFrame global: baseline_folds_metrics
          # ------------------------------------------------------------
          if ("baseline_folds_metrics" in globals()) and (baseline_folds_metrics is not None):

              # (opcional pero recomendado) asegurar que exista la fila
              if model_key not in baseline_folds_metrics.index:
                  baseline_folds_metrics.loc[model_key, :] = None

              for split, m in [("train", metrics_tr), ("valid", metrics_va), ("test", metrics_te)]:
                  for k, v in m.items():
                      baseline_folds_metrics.loc[model_key, f"{split}_{k}"] = v

          # ------------------------------------------------------------
          # 10) Guardar en RAM
          # ------------------------------------------------------------
          models[fold]     = model_fold
          scalers_y[fold]  = scaler_y_fold
          ytr_pred_d[fold] = ytr_pred
          yva_pred_d[fold] = yva_pred
          yte_pred_d[fold] = yte_pred

          # ------------------------------------------------------------
          # 11) Checkpoint en disco
          # ------------------------------------------------------------
          ruta_ckpt = path_models[fold]["model_path"]

          checkpoint = {
              "model_state":   model_fold.state_dict(),
              "encoder_state": encoder_fold.state_dict(),
              "head_state":    head_fold.state_dict(),
              "scaler_y":      scaler_y_fold,
              "metrics_train": metrics_tr,
              "metrics_valid": metrics_va,
              "metrics_test":  metrics_te,
              "hparams": {
                  "T": T,
                  "F": F,
                  "lr": baseline_params['lr'],
                  "weight_decay": baseline_params['weight_decay'],
                  "max_epochs": baseline_params['max_epochs'],
                  "patience": baseline_params['patience'],
                  "grad_clip": baseline_params['grad_clip'],
                  "use_amp": True,
                  "pooling": baseline_params['pooling'],
                  "batch_size": batch_train,
              },
          }

          torch.save(checkpoint, ruta_ckpt)
          print(f"✔ Checkpoint guardado para fold {fold} en: {ruta_ckpt}")

          # ------------------------------------------------------------
          # 12) Limpieza explícita de memoria (GPU + RAM)
          # ------------------------------------------------------------
          import gc

          del dl_tr_fold, dl_va_fold
          del ytr_pred, yva_pred, yte_pred

          torch.cuda.empty_cache()
          gc.collect()

flag_baseline_metrics=False → se inicia entrenamiento por folds.

=== Entrenando modelo: baseline_fold_1 ===
Epoch 001  train=2.617689e-01  valid=6.229209e-01
Epoch 002  train=2.276641e-01  valid=6.390729e-01
Epoch 003  train=2.191224e-01  valid=6.410610e-01
Epoch 004  train=2.148598e-01  valid=6.296703e-01
Epoch 005  train=2.102946e-01  valid=6.407632e-01
Epoch 006  train=2.082878e-01  valid=6.197260e-01
Epoch 007  train=2.067606e-01  valid=6.252935e-01
Epoch 008  train=2.031295e-01  valid=6.317324e-01
Epoch 009  train=1.994542e-01  valid=6.270877e-01
Epoch 010  train=1.987699e-01  valid=6.091316e-01
Epoch 011  train=1.960362e-01  valid=6.049724e-01
Epoch 012  train=1.928536e-01  valid=6.308941e-01
Epoch 013  train=1.903284e-01  valid=6.359033e-01
Epoch 014  train=1.889278e-01  valid=6.225358e-01
Epoch 015  train=1.867124e-01  valid=6.159242e-01
Epoch 016  train=1.863393e-01  valid=6.273414e-01
Epoch 017  train=1.821265e-01  valid=6.255336e-01
Epoch 018  train=1.797597e-01  valid=6.71

KeyboardInterrupt: 

In [74]:
baseline_folds_metrics

,RMSE,MAE,R2,SMAPE,DirAcc,train_RMSE,train_MAE,train_R2,train_SMAPE,train_DirAcc,valid_RMSE,valid_MAE,valid_R2,valid_SMAPE,valid_DirAcc,test_RMSE,test_MAE,test_R2,test_SMAPE,test_DirAcc
baseline_fold_1,None,None,None,None,None,0.002178,0.001571,0.818307,80.846117,0.831524,0.003974,0.002771,0.544396,91.205083,0.807133,0.004990,0.002768,0.439988,114.775035,0.782745
baseline_fold_2,None,None,None,None,None,0.002837,0.001978,0.707743,86.528367,0.815602,0.002658,0.001908,0.613347,87.580046,0.819544,0.004877,0.002639,0.465220,111.475016,0.794270
baseline_fold_3,None,None,None,None,None,0.002088,0.001543,0.833672,77.538958,0.841201,0.002011,0.001495,0.665171,89.197566,0.809262,0.004533,0.002611,0.537832,109.807708,0.772727
baseline_fold_4,None,None,None,None,None,0.002500,0.001774,0.744314,83.380920,0.822865,0.002080,0.001416,0.613171,91.450153,0.801992,0.004511,0.002558,0.542349,104.496034,0.764541


## 9. Métricas

In [ ]:
if flag_baseline_folds_metrics:
    print("flag_baseline_folds_metrics=True → ya existen métricas del entrenamiento.")
else:
    print("flag_baseline_folds_metrics=False → se inicia entrenamiento por folds.")
    cols_base = ["RMSE", "MAE", "R2", "SMAPE", "DirAcc"]
    # columnas que efectivamente existen en el DataFrame
    cols_to_drop = [c for c in cols_base if c in baseline_folds_metrics.columns]
    # eliminar solo si hay columnas para eliminar
    if cols_to_drop:
        baseline_folds_metrics = baseline_folds_metrics.drop(columns=cols_to_drop)

In [ ]:
baseline_folds_metrics

In [ ]:
if flag_baseline_folds_metrics == False:
  save_metrics(baseline_folds_metrics, "5_3_baseline_model","0_baseline_folds_metrics")
else:
  print("Ya existen métricas del entrenamiento y están guardadas en disco")

### 9.1. Análisis de primeros resultados

1. Consistencia del desempeño entre folds

    - El modelo presenta un comportamiento estable en todas las particiones temporales.
    - Las métricas de entrenamiento y validación muestran poca variación.
    - Los valores de RMSE en entrenamiento se encuentran entre 0.00166 y 0.00212, mientras que el RMSE de validación se mantiene entre 0.00185 y 0.00196.
    - Esta estabilidad indica que el modelo captura patrones generales del mercado sin depender de particularidades de cada fold.

2. Diferencias claras entre Train, Valid y Test

    - En todos los folds se observa una caída en el rendimiento cuando se evalúa sobre el conjunto de test.
    - El RMSE y el MAE aumentan de forma consistente en test, mientras que el R² disminuye.
    - La Directional Accuracy se mantiene cercana al 0.80, aunque también muestra una leve reducción respecto a Train y Valid.
    - Este comportamiento es coherente con:
      - La no estacionariedad de los datos financieros intradía.
      - Posibles cambios de régimen en los días reservados para test.
      - Patrones aprendidos por el modelo que no necesariamente se repiten en el futuro.

3. Relación entre Train y Test (RMSE aproximadamente 2.3 veces mayor)

    - Por ejemplo, en el fold 1:
      - Train RMSE: 0.001869
      - Test RMSE: 0.004352

    - El incremento del error indica un nivel moderado de sobreajuste.
    - Aun así, el modelo mantiene capacidad predictiva en términos direccionales (Direction Accuracy alrededor de 0.80).

4. Comportamiento del R²

    - Los valores promedio aproximados de R² son:
      - Entrenamiento: alrededor de 0.85
      - Validación: entre 0.67 y 0.70
      - Test: entre 0.54 y 0.58
    - Aunque el R² disminuye en test, estos valores son razonables considerando el alto nivel de ruido y variabilidad de las series intradía.
    - Un R² en el orden del 50 % resulta aceptable en este tipo de problemas.

5. SMAPE estable entre folds

    - Los valores observados de SMAPE son:
      - Train: entre 72 y 78
      - Valid: entre 85 y 89
      - Test: entre 89 y 96
    - La diferencia entre validación y test es relativamente pequeña, lo que indica que la dificultad del horizonte de predicción se mantiene consistente.

6. Dirección de movimiento (DirAcc) elevada en Test

    - La precisión direccional (Direction Accuracy) se mantiene entre 0.79 y 0.82.
    - Este desempeño es especialmente relevante para aplicaciones donde la predicción del signo del retorno resulta más importante que el valor exacto.

**Conclusión**

- El modelo muestra un desempeño sólido y consistente en los conjuntos de entrenamiento y validación.
- En el conjunto de test se observa un descenso esperado debido a la naturaleza no estacionaria del mercado, aunque la performance sigue siendo estable.
- La consistencia entre folds sugiere que el modelo captura relaciones reales presentes en los datos intradía.
- La reducción del R² y el aumento del error en test reflejan un sobreajuste moderado o la presencia de cambios de régimen en el mercado.
- Una Direction Accuracy cercana al 80 % posiciona al modelo Transformer como un candidato competitivo para tareas de predicción direccional de retornos intradía.

### 9.2. Promedio ponderado por cantidad de muestras de cada conjunto (train, valid, test).

En nuestro proyecto de series temporales, cada fold cuenta con el mismo número de ventanas de train, valid y test:

    - `w_train`: 223871
    - `w_valid`: 24898
    - `w_test`: 27852

Por lo cual, nuestros pesos son iguales en todos los folds. Esto ocurre porque usamos K folds sobre días completos, pero la generación de ventanas produce exactamente el mismo número de muestras por día, por lo que cada fold conserva la misma distribución.

Aunque cada fold tenga el mismo número de ventanas, cada conjunto dentro del fold no debe tener la misma importancia.

Un promedio que mezcle `train`, `valid` y `test` sin ponderar, da el mismo peso a métricas que representan cosas distintas.

El propósito del conjunto:

- Train: mide ajuste del modelo
- Valid: guía selección de hiperparámetros
- Test: mide capacidad de generalización

Es metodológicamente incorrecto darle el mismo peso a los tres, porque no cumplen la misma función.

La ponderación respeta el volumen real de datos usados en cada split, no su rol en el pipeline.

In [ ]:
pesos_folds

FUNCIÓN PARA PROMEDIO PONDERADO DE MÉTRICAS POR FOLD


In [ ]:
def weighted_avg_metrics_from_df(df, w_train, w_valid, w_test):
    """
    Calcula el promedio ponderado de métricas a partir de un DataFrame
    con columnas del tipo train_RMSE, valid_RMSE, test_RMSE, etc.

    df : DataFrame con un fold por fila
    w_train, w_valid, w_test : pesos (cantidad de muestras por conjunto)
    """

    # Métricas base
    metricas = ["RMSE", "MAE", "R2", "SMAPE", "DirAcc"]

    resultados = {}

    for m in metricas:
        col_train = f"train_{m}"
        col_valid = f"valid_{m}"
        col_test  = f"test_{m}"

        # Promedio ponderado por fold, luego promedio entre folds
        valores_fold = (
            df[col_train] * w_train +
            df[col_valid] * w_valid +
            df[col_test]  * w_test
        ) / (w_train + w_valid + w_test)

        # Promedio total final entre folds
        resultados[m] = valores_fold.mean()

    return resultados


Aunque los folds tengan la misma cantidad de ventanas, cada split dentro del fold no tiene igual tamaño:

In [ ]:
pesos_folds

Si no los ponderamos, estaríamos diciendo implícitamente:

- “El error en train y el error en test valen lo mismo”, aunque train tiene 10 veces más muestras que valid/test. **Eso sería una distorsión estadística.**

El promedio ponderado refleja que:
- El error de train afecta más el resultado global porque está medido sobre más muestras.
- El error de valid y test aportan menos porque su volumen relativo es menor

#### 9.2.1. Aplicación de ponderado

In [ ]:
if flag_baselines_metrics:
    print("metrics_baselines=True → se omite el ponderado porque ya existe el dataset.")
else:
    print("metrics_baselines=False → se inicia el ponderado")
    for k in k_folds:
        w = pesos_folds[k]

        # promedio ponderado para ESTE fold (sale como dict)
        res = weighted_avg_metrics_from_df(
            transformers_folds_metrics.loc[[f"transformer_fold_{k}"]],
            w["w_train"],
            w["w_valid"],
            w["w_test"]
        )

        # índice correspondiente en transformers_metrics
        idx = f"transformer_fold_{k}"

        # escribir directamente en el dataset transformers_metrics
        transformers_baseline_metrics.loc[idx, cols_base] = [res[m] for m in cols_base]

In [ ]:
transformers_baseline_metrics

In [ ]:
if flag_baselines_metrics == False:
  save_metrics(transformers_baseline_metrics,"5_3_transformer_baseline","1_transformers_baseline_metrics")
else:
  print("Ya existen métricas del entrenamiento y están guardadas en disco")

**Análisi sobre los resultados de las métricas ponderadas**

1. El desempeño del modelo es estable entre folds.

    Las métricas RMSE, MAE y R² presentan variaciones pequeñas, lo que indica que el modelo mantiene un comportamiento consistente bajo distintas particiones temporales del dataset.

2. El error absoluto es bajo y homogéneo.

    El MAE se encuentra aproximadamente entre 0.00138 y 0.00160, lo que refleja que el modelo logra una precisión adecuada en la escala de retornos intradía.

3. El R² muestra una capacidad explicativa sólida.

    Los valores entre 0.77 y 0.83 sugieren que el modelo captura una proporción significativa de la variabilidad del retorno a predecir.
    El fold 5 muestra un desempeño levemente inferior (0.776), posiblemente por condiciones de mercado distintas en ese periodo.

4. La Directional Accuracy (DirAcc) es consistentemente alta.

    Con valores entre 0.833 y 0.846, el modelo demuestra una fuerte capacidad para anticipar correctamente la dirección del próximo movimiento del MNQ.
    Este comportamiento es especialmente relevante para aplicaciones operativas basadas en señales direccionales.

5. El SMAPE indica un error porcentual moderado pero estable.

    Los valores entre 75 y 80 reflejan un nivel de error relativo acorde a la volatilidad intrínseca del instrumento; no se observan desviaciones fuertes entre folds.

6. No se identifican signos de inestabilidad o sensibilidad excesiva al split.

    La variación entre métricas es limitada, lo cual sugiere que el modelo generaliza razonablemente bien dentro del esquema de validación.

**Conclusión**:

El modelo Transformer muestra desempeño sólido y consistente en todos los folds. Las métricas de error son bajas, la capacidad explicativa (R²) es elevada para un problema de series intradía, y la precisión direccional supera el 83 %, lo cual confirma que el modelo captura patrones relevantes del comportamiento futuro del MNQ.